In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0
)

In [2]:
from typing import TypedDict,Annotated,List,Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel,Field
from langgraph.graph import END,StateGraph
from typing import List
import operator

#### Orchestrator-Worker Pattern

##### Structured Output

In [3]:
# dish schema for a single dish
class Dish(BaseModel):
    name:str=Field(
        description="Name of the dish (for example, Spaghetti Bolognese, Chicken Curry)"
    )
    ingredients:List[str]=Field(
        description="List of the ingredients needed for this dish, separated by commas"
    )
    location:str=Field(
        description="The cuisine or cultural origin of the dish(for example italy,indian,nepal)"
    )
    

In [4]:
# dish schema for for a list of Dish object
class Dishes(BaseModel):
    sections:List[Dish]=Field(
        description="A list of gorcery sections, one for each dish, with ingedients"
    )

In [5]:
# construct a prompt template
dish_prompt=ChatPromptTemplate.from_messages([(
    "system",
    "you are an assistant that generates a structured grocery list.\n\n"
    "the user wants to prepare the following meals: {meals}\n\n"
    "for each meals, return a section with:\n"
    "- the name of the dish\n"
    "- a comma-separated list of ingredients needed for that dish.\n"
    "- the cusine or cultural origin of the food"
)
    
])

In [6]:
import re

# use LCEL to pipe the prompt to an LLM with a structured output of Dishes
planner_pipe = dish_prompt | llm.with_structured_output(Dishes) # use llm.with_structured_output(Dishes)
response = planner_pipe.invoke({"meals": "carrot cake"})

In [7]:
from pprint import pprint
pprint(response)

Dishes(sections=[Dish(name='Carrot Cake', ingredients=['1 cup all-purpose flour', '1 cup sugar', '1/2 cup unsalted butter', '1/2 cup milk', '1 egg', '1/2 teaspoon vanilla extract', '1 cup carrots (sliced)', '1/2 cup cake mix (or 1/2 cup flour, 1/2 cup sugar, 1/2 cup butter, 1/2 cup milk, 1/2 teaspoon vanilla extract)'], location='Western')])


#### State(Orchestrations)

In [8]:
class State(TypedDict):
    meals:str
    sections: List[Dish]
    completed_menu: Annotated[List[str],operator.add]
    final_meal_guide:str
    

In [9]:
dummy_state: State={
    "meals":"Spaghetti Bolognese and chicken Stir fry",
    "sections":[],
    "completed_menu":[],
    "final_meal_guide": ""
    
}
report_sections=planner_pipe.invoke({"meals":dummy_state['meals']})

In [10]:
report_sections

Dishes(sections=[Dish(name='Spaghetti Bolognese', ingredients=['2 pounds ground beef', '1 cup chopped tomatoes', '1 onion', '2 cloves garlic', '1 tablespoon olive oil', '1 teaspoon salt', '1 teaspoon black pepper', '1/2 cup chopped fresh parsley', '1/2 cup chopped fresh basil', '1/2 cup grated Parmesan cheese'], location='Italian'), Dish(name='Chicken Stir Fry', ingredients=['400 grams chicken breast or thighs', '2 tablespoons soy sauce', '1 tablespoon garlic', '1 tablespoon ginger', '1/2 cup vegetable broth', '1/4 cup rice', '1/2 teaspoon salt'], location='Chinese')])

In [11]:
for i, section in enumerate(report_sections.sections):
    print(f"Dish {i+1}\n")
    # add each dish to our dummy state
    dummy_state["sections"].append(section)
    print(f"Item Name: {section.name}")
    print(f"Location/Cuisine: {section.location}")
    print(f"Ingredients: {', '.join(section.ingredients)}.")

Dish 1

Item Name: Spaghetti Bolognese
Location/Cuisine: Italian
Ingredients: 2 pounds ground beef, 1 cup chopped tomatoes, 1 onion, 2 cloves garlic, 1 tablespoon olive oil, 1 teaspoon salt, 1 teaspoon black pepper, 1/2 cup chopped fresh parsley, 1/2 cup chopped fresh basil, 1/2 cup grated Parmesan cheese.
Dish 2

Item Name: Chicken Stir Fry
Location/Cuisine: Chinese
Ingredients: 400 grams chicken breast or thighs, 2 tablespoons soy sauce, 1 tablespoon garlic, 1 tablespoon ginger, 1/2 cup vegetable broth, 1/4 cup rice, 1/2 teaspoon salt.


##### Orchestrator Node

In [12]:
def orchestrator(state:State):
    """Orhcestrator that generate a structured dish list form the given meals."""
    #use the planner_pipe llm to break the user's meal list into structured dish sections
    dish_description=planner_pipe.invoke({"meals":state['meals']})
    # return the list of dish sections to be passed to worker nodes
    return {"sections":dish_description.sections}

##### Worker Node

In [13]:
chef_prompt=ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a world class chef from {location}.\n\n"
        "Please introduce yourself briefly and present a detailed walkthrough for preparing the dish:{name}.\n"
        "Your response should include:\n"
        "- Start with hello with your name and culinary background\n"
        "- A clear list of preparations steps\n"
        "- A full explanations of the cooking process\n\n"
        "Use the following ingredients: {ingredients}"
    )
])

In [14]:
chef_pipe=chef_prompt|llm

In [15]:
class WorkerState(TypedDict):
    section:Dish
    completed_menu:Annotated[list,operator.add]

In [16]:
def assign_workers(state:State):
    """Assign a worker to each section in the plan"""
    # Kick off section writing in parallel via Send() API
    return [Send("chef_worker",{"section":s}) for s in state['sections']]    

In [17]:
def chef_worker(state: WorkerState):
    """Worker node that generates the cooking instructions for one meal section."""

    # Use the language model to generate a meal preparation plan
    # The model receives the dish name, location, and ingredients from the current section
    meal_plan = chef_pipe.invoke({
        "name": state["section"].name,
        "location": state["section"].location,
        "ingredients": state["section"].ingredients
    })

    # Return the generated meal plan wrapped in a list under completed_sections
    # This will be merged into the main state using operator.add in LangGraph
    return {"completed_menu": [meal_plan.content]}


In [ ]:
dummy_dishes: List[Dish] = dummy_state["sections"]

# simulate LangGraph's fan-out and merging behavior
for section in dummy_dishes:
    # construct individual WorkerState
    worker_state: WorkerState = {
        "section": section,
        "recipe": []  # LangGraph merges this later
    }

    # call the worker logic directly
    result = chef_worker(worker_state)

    # merge the result into combined menu (LangGraph would do this with operator.add)
    dummy_state["completed_menu"] += result["completed_menu"]

In [ ]:
completed_menu_sections = "\n".join(dummy_state["completed_menu"])
print(completed_menu_sections[:1000])

In [ ]:
def synthesizer(state: State):
    """Synthesize full report from sections"""

    # list of completed sections
    completed_sections = state["completed_menu"]

    # format completed section to str to use as context for final sections
    completed_menu = "\n\n---\n\n".join(completed_sections)

    return {"final_meal_guide": completed_menu}